In [16]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd

In [17]:
## Loading the Models
model = load_model('Models\model.h5')

## Loading Encoders and scalers
with open('Models/onehot_encoder.pkl', 'rb') as file1:
    onehot_geo = pickle.load(file1)

with open('Models/label_encoder.pkl', 'rb') as file2:
    lb_gender = pickle.load(file2)

with open('Models/scaker.pkl', 'rb') as file3:
    scaler = pickle.load(file3)

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\singh\AppData\Local\Temp\ipykernel_10760\1713936835.py:2: SyntaxWarning: invalid escape sequence '\m'
  model = load_model('Models\model.h5')


In [18]:
## Example Input data
input_data = {
    'CreditScore' : 600,
    "Geography" : 'France',
    "Gender" : "Male",
    "Age" : 40,
    "Tenure" : 3,
    "Balance" : 60000,
    "NumOfProducts" : 2,
    "HasCrCard" : 1,
    "IsActiveMember" : 1,
    "EstimatedSalary" : 50000
}

In [20]:
# Step 1: Convert dict to DataFrame
input_df = pd.DataFrame([input_data])

# Step 2: Encode Geography
geo_encoded = onehot_geo.transform(input_df[['Geography']])
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_geo.get_feature_names_out(['Geography'])
)

# Step 3: Merge
input_df = pd.concat([input_df.reset_index(drop=True), geo_encoded_df], axis=1)

# Step 4: Drop original column
input_df.drop(columns=['Geography'], inplace=True)

In [21]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [22]:
input_df['Gender'] = lb_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [23]:
## Applying Scaling
input_scaled = scaler.transform(input_df)

In [25]:
prediction = model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step


array([[0.03496424]], dtype=float32)

In [27]:
prediction_proba = prediction[0][0]
prediction_proba

np.float32(0.03496424)

In [28]:
if prediction_proba > 0.5:
    print("The Customer is Likely to Churn")
else:
    print("The Customer is not Likely to Churn")

The Customer is not Likely to Churn
